# This Notebook contains codes to run DS & ML models on local windows machine.

Prepare for Hybrid labelling (iterative human labelling with genAI services)

In [ ]:
def clean_illegal_chars(text):
    if isinstance(text, str):
        return re.sub(r"[\x00-\x1F\x7F-\x9F]", "", text)
    return text

# Apply to all string columns
final_news_sampled_df = final_news_sampled_df.applymap(clean_illegal_chars)

# Now export safely
final_news_sampled_df.to_excel('sample_to_label.xlsx', index = False, engine="openpyxl")

After labelling this sample, I will load them in and analyse

### Traditional DS Models - Classification

In this section, I considered logistic regression, random forest and XGBoost for classifying the articles. I also did hyperparameter tuning using 5-fold cross validation. 

In [41]:
import pandas as pd
import numpy as np
import re
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, make_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


# ----------- Custom Transformers -----------

class TextSelector(BaseEstimator, TransformerMixin):
    def __init__(self, key):
        self.key = key

    def fit(self, X, y=None): return self

    def transform(self, X):
        return X[self.key].fillna("")


class NormalizedKeywordCountExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, keywords, column):
        self.keywords = keywords
        self.column = column

    def fit(self, X, y=None): return self

    def transform(self, X):
        col = X[self.column].fillna("")
        pattern = '|'.join(re.escape(kw) for kw in self.keywords)

        def keyword_density(text):
            words = text.split()
            word_count = len(words) if len(words) > 0 else 1
            keyword_count = len(re.findall(pattern, text, flags=re.IGNORECASE))
            return keyword_count / word_count

        densities = col.apply(keyword_density)
        return pd.DataFrame({f"{self.column}_keyword_density": densities})


# ----------- Main Evaluator Class -----------

class NewsClassifierEvaluator:
    def __init__(self, df, label_col, tfidf_cols, keyword_cols, keywords=None, test_size=0.2, random_state=42):
        self.df = df.copy()
        self.label_col = label_col
        self.tfidf_cols = tfidf_cols
        self.keyword_cols = keyword_cols
        self.keywords = keywords or ["fraud", "money laundering", "corruption", "sanction", "embezzlement"]
        self.test_size = test_size
        self.random_state = random_state

        self.models = {
            'LogisticRegression': LogisticRegression(max_iter=1000),
            'RandomForest': RandomForestClassifier(),
            'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss')
        }

        self.param_grids = {
            'LogisticRegression': {
                'clf__C': [0.01, 0.1, 1, 10]
            },
            'RandomForest': {
                'clf__n_estimators': [100, 200],
                'clf__max_depth': [None, 10, 20]
            },
            'XGBoost': {
                'clf__n_estimators': [100, 200],
                'clf__max_depth': [3, 5, 7],
                'clf__learning_rate': [0.05, 0.1]
            }
        }

        self.X_train, self.X_test, self.y_train, self.y_test = self._train_test_split()

    def _train_test_split(self):
        return train_test_split(
            self.df,
            self.df[self.label_col],
            test_size=self.test_size,
            stratify=self.df[self.label_col],
            random_state=self.random_state
        )

    def _evaluate_model(self, y_true, y_pred, y_proba):
        return {
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred),
            'Recall': recall_score(y_true, y_pred),
            'F1 Score': f1_score(y_true, y_pred),
            'ROC AUC': roc_auc_score(y_true, y_proba)
        }

    def _build_pipeline(self, model):
        feature_extractors = []

        for col in self.tfidf_cols:
            feature_extractors.append((
                f"{col}_tfidf",
                Pipeline([
                    ('select', TextSelector(col)),
                    ('tfidf', TfidfVectorizer(max_features=3000, stop_words='english'))
                ])
            ))

        for col in self.keyword_cols:
            feature_extractors.append((
                f"{col}_keyword_density",
                NormalizedKeywordCountExtractor(self.keywords, column=col)
            ))

        return Pipeline([
            ('features', FeatureUnion(feature_extractors)),
            ('clf', model)
        ])

    def cross_validate_model(self, model_name, k=5, tune=False):
        model = self.models[model_name]
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=self.random_state)
        metrics_list = []

        for fold, (train_idx, val_idx) in enumerate(skf.split(self.X_train, self.y_train), 1):
            train_df = self.X_train.iloc[train_idx]
            val_df = self.X_train.iloc[val_idx]
            y_train_fold = self.y_train.iloc[train_idx]
            y_val_fold = self.y_train.iloc[val_idx]

            pipeline = self._build_pipeline(model)

            if tune:
                param_grid = self.param_grids[model_name]
                grid = GridSearchCV(pipeline, param_grid, scoring='f1', cv=3, n_jobs=-1)
                grid.fit(train_df, y_train_fold)
                best_pipeline = grid.best_estimator_
            else:
                pipeline.fit(train_df, y_train_fold)
                best_pipeline = pipeline

            y_pred = best_pipeline.predict(val_df)
            y_proba = best_pipeline.predict_proba(val_df)[:, 1]

            metrics = self._evaluate_model(y_val_fold, y_pred, y_proba)
            metrics['Fold'] = fold
            metrics_list.append(metrics)

        return pd.DataFrame(metrics_list)

    def evaluate_on_holdout(self, model_name, tune=False):
        model = self.models[model_name]
        pipeline = self._build_pipeline(model)

        if tune:
            param_grid = self.param_grids[model_name]
            grid = GridSearchCV(pipeline, param_grid, scoring='f1', cv=3, n_jobs=-1)
            grid.fit(self.X_train, self.y_train)
            pipeline = grid.best_estimator_
        else:
            pipeline.fit(self.X_train, self.y_train)

        y_pred = pipeline.predict(self.X_test)
        y_proba = pipeline.predict_proba(self.X_test)[:, 1]

        return self._evaluate_model(self.y_test, y_pred, y_proba)


# ----------- Example Usage -----------

if __name__ == '__main__':
    df = pd.read_excel("./sample_labelled/sample_labelled.xlsx", engine="openpyxl")

    evaluator = NewsClassifierEvaluator(
        df=df,
        label_col='adverse_news_financial_crime_scandal_sanctions',
        tfidf_cols=['title', 'description', 'clean_full_text'],
        keyword_cols=['title', 'description','clean_full_text'],
        keywords=[
            "fraud", "money laundering", "corruption", "bribery",
            "sanction", 'ponzi', 'pyramid scheme',
            'insider trading', 'terrorist financing', 'tax-evasion'
        ]
    )

    evaluation_df = pd.DataFrame()

    for model_name in evaluator.models:
        print(f"\nTuned cross-validation results for {model_name}:")
        cv_results = evaluator.cross_validate_model(model_name, tune=True)
        cv_results['model'] = model_name
        cv_results['type_of_eval'] = 'CV'
        evaluation_df = pd.concat((evaluation_df, cv_results), ignore_index=True)
        print(cv_results.drop(columns='Fold').mean(numeric_only=True))

        print(f"\nTuned hold-out evaluation for {model_name}:")
        holdout_results = evaluator.evaluate_on_holdout(model_name, tune=True)
        holdout_results['model'] = model_name
        holdout_results['type_of_eval'] = 'test/hold-out'
        evaluation_df = pd.concat((evaluation_df, pd.DataFrame([holdout_results])), ignore_index=True)
        print(holdout_results)


    evaluation_df.to_excel('evaluation/evaluation_simple_classifiers.xlsx', index = False)


Tuned cross-validation results for LogisticRegression:
Accuracy     0.899167
Precision    0.858850
Recall       0.664322
F1 Score     0.748034
ROC AUC      0.958859
dtype: float64

Tuned hold-out evaluation for LogisticRegression:
{'Accuracy': 0.92, 'Precision': 0.8728813559322034, 'Recall': 0.7573529411764706, 'F1 Score': 0.8110236220472441, 'ROC AUC': np.float64(0.9651527636916835), 'model': 'LogisticRegression', 'type_of_eval': 'test/hold-out'}

Tuned cross-validation results for RandomForest:
Accuracy     0.835000
Precision    0.805980
Recall       0.357917
F1 Score     0.494230
ROC AUC      0.893839
dtype: float64

Tuned hold-out evaluation for RandomForest:
{'Accuracy': 0.84, 'Precision': 0.75, 'Recall': 0.4411764705882353, 'F1 Score': 0.5555555555555556, 'ROC AUC': np.float64(0.9077633747464503), 'model': 'RandomForest', 'type_of_eval': 'test/hold-out'}

Tuned cross-validation results for XGBoost:


C:\Users\angsi\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:15:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\angsi\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:17:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\angsi\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:20:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\angsi\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:23:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtr

Accuracy     0.885000
Precision    0.811224
Recall       0.640112
F1 Score     0.714989
ROC AUC      0.934863
dtype: float64

Tuned hold-out evaluation for XGBoost:


C:\Users\angsi\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:29:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


{'Accuracy': 0.9116666666666666, 'Precision': 0.8487394957983193, 'Recall': 0.7426470588235294, 'F1 Score': 0.792156862745098, 'ROC AUC': np.float64(0.9522771932048681), 'model': 'XGBoost', 'type_of_eval': 'test/hold-out'}


### Topic Modelling using BerTopics + MLP Classifier based on BerTopics + other features

Steps I did in the following code:
1. Do a train, validation, test split (60-20-20).
2. Use the clean_full_text column as text column to get TF-IDF, use title and description for pre-defined keyword counts (normalised, i.e. divided by total number of tokens) occurring within the title and description
3. Run BerTopics Topic Modelling then get top topics, iteratively check cosine similarity of these topics with the pre-defined keywords until we get at least 5 topics which are similar to our pre-defined keywords relating to financial crime/sanctions/scandals.
4. Use those features created in steps 2 and 3, together with the labels, using the embeddings in BerTopics model, train a MLP

I believe the text is too noisy, so it obscures signals from the data and more work is required for feature engineering

In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer, util
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from typing import List

try:
    from bertopic import BERTopic
    BERTopic_AVAILABLE = True
except ImportError:
    BERTopic_AVAILABLE = False


# ----- Feature Extractor Classes -----

class TextSelector(BaseEstimator, TransformerMixin):
    def __init__(self, column):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.column].fillna("")


class KeywordDensityExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, keywords, column):
        self.keywords = keywords
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return pd.DataFrame([
            [text.lower().count(kw.lower()) / (len(text.split()) + 1e-6) for kw in self.keywords]
            for text in X[self.column].fillna("")
        ])


class TopicFeatureAdder(BaseEstimator, TransformerMixin):
    def __init__(self, topic_model: BERTopic, text_column: str, relevant_topic_ids: List[int]):
        self.topic_model = topic_model
        self.text_column = text_column
        self.relevant_topic_ids = relevant_topic_ids

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        texts = X[self.text_column].fillna("").tolist()
        topics, _ = self.topic_model.transform(texts)
        return pd.DataFrame([[int(t in self.relevant_topic_ids)] for t in topics], index=X.index)


# ----- PyTorch Dataset -----

class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


# ----- Neural Network Classifier -----

class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.model(x)


# ----- Main Classifier Pipeline -----

class BerTopicsEnhancedClassifier:
    def __init__(self, df: pd.DataFrame, label_col: str, text_col: str = 'clean_full_text',
                 keyword_cols: List[str] = None, keywords: List[str] = None,
                 use_tfidf: bool = True, use_bertopic: bool = True):
        self.df = df.copy()
        self.label_col = label_col
        self.text_col = text_col
        self.keyword_cols = keyword_cols or ['title', 'description']
        self.keywords = keywords or [
            "fraud", "money laundering", "corruption", "bribery",
            "sanction", "ponzi", "pyramid scheme",
            "insider trading", "terrorist financing", "tax-evasion"
        ]
        self.use_tfidf = use_tfidf
        self.use_bertopic = use_bertopic and BERTopic_AVAILABLE

        self.df['topic_text'] = self.df[self.keyword_cols].fillna("").agg(" ".join, axis=1)
        self.embedding_model = SentenceTransformer("all-MiniLM-L12-v2")
        self._split_data()
        if self.use_bertopic:
            self._initialize_topic_model_with_relevance_filter()

    def _split_data(self):
        train_val_df, self.test_df = train_test_split(
            self.df, test_size=0.2, stratify=self.df[self.label_col], random_state=42)
        self.train_df, self.val_df = train_test_split(
            train_val_df, test_size=0.25, stratify=train_val_df[self.label_col], random_state=42)

    def _initialize_topic_model_with_relevance_filter(self):
        current_train_texts = self.train_df['topic_text'].tolist()
        min_relevant = 5
        iteration = 0
        relevant_topic_ids = []

        while len(relevant_topic_ids) < min_relevant and iteration < 5:
            self.topic_model = BERTopic(embedding_model=self.embedding_model)
            topics, _ = self.topic_model.fit_transform(current_train_texts)
            topic_info = self.topic_model.get_topic_info()
            topic_ids = topic_info['Topic'].tolist()
            topic_keywords = [" ".join([w for w, _ in self.topic_model.get_topic(t)[:5]])
                              for t in topic_ids if t != -1]

            topic_embeddings = self.embedding_model.encode(topic_keywords, convert_to_tensor=True)
            keyword_embeddings = self.embedding_model.encode(self.keywords, convert_to_tensor=True)
            similarities = util.cos_sim(topic_embeddings, keyword_embeddings)

            relevant_topic_ids = [topic_ids[i] for i in range(len(topic_ids))
                                  if i < similarities.shape[0] and similarities[i].max().item() > 0.4 and topic_ids[i] != -1]
            iteration += 1

        self.relevant_topic_ids = relevant_topic_ids

    def preprocess_features(self, df, fit=True):
        feature_extractors = []
        if self.use_tfidf:
            feature_extractors.append((
                "tfidf", Pipeline([
                    ('select', TextSelector(self.text_col)),
                    ('tfidf', TfidfVectorizer(max_features=3000, stop_words='english'))
                ])
            ))

        for col in self.keyword_cols:
            feature_extractors.append((
                f"{col}_keywords",
                KeywordDensityExtractor(self.keywords, column=col)
            ))

        if self.use_bertopic:
            feature_extractors.append((
                "topic", TopicFeatureAdder(self.topic_model, text_column='topic_text',
                                           relevant_topic_ids=self.relevant_topic_ids)
            ))

        self.feature_pipeline = FeatureUnion(feature_extractors)

        base_features = self.feature_pipeline.fit_transform(df) if fit else self.feature_pipeline.transform(df)

        if hasattr(base_features, "toarray"):
            base_features = base_features.toarray()
        if base_features.ndim == 1:
            base_features = base_features.reshape(-1, 1)

        embeddings = self.embedding_model.encode(df[self.text_col].fillna("").tolist(), show_progress_bar=True)
        embeddings = np.array(embeddings)
        if embeddings.ndim == 1:
            embeddings = embeddings.reshape(-1, 1)

        combined = np.hstack([base_features, embeddings])

        if fit:
            self.scaler = StandardScaler()
            combined_scaled = self.scaler.fit_transform(combined)
        else:
            combined_scaled = self.scaler.transform(combined)

        return combined_scaled

    def train(self, epochs=10, batch_size=32, lr=1e-3):
        X_train = self.preprocess_features(self.train_df, fit=True)
        y_train = self.train_df[self.label_col].values
        X_val = self.preprocess_features(self.val_df, fit=False)
        y_val = self.val_df[self.label_col].values

        train_loader = DataLoader(FeatureDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(FeatureDataset(X_val, y_val), batch_size=batch_size)

        model = MLPClassifier(X_train.shape[1])
        self.model = model
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        self.criterion = nn.CrossEntropyLoss()

        for epoch in range(epochs):
            model.train()
            running_train_loss = 0.0
            for X_batch, y_batch in train_loader:
                self.optimizer.zero_grad()
                out = model(X_batch)
                loss = self.criterion(out, y_batch)
                loss.backward()
                self.optimizer.step()
                running_train_loss += loss.item() * X_batch.size(0)

            epoch_train_loss = running_train_loss / len(train_loader.dataset)

            model.eval()
            running_val_loss = 0.0
            with torch.no_grad():
                for X_val_batch, y_val_batch in val_loader:
                    outputs = model(X_val_batch)
                    val_loss = self.criterion(outputs, y_val_batch)
                    running_val_loss += val_loss.item() * X_val_batch.size(0)
            epoch_val_loss = running_val_loss / len(val_loader.dataset)

            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {epoch_train_loss:.4f} - Val Loss: {epoch_val_loss:.4f}")

    def evaluate(self):
        from sklearn.metrics import classification_report

        X_test = self.preprocess_features(self.test_df, fit=False)
        y_test = self.test_df[self.label_col].values
        test_dataset = FeatureDataset(X_test, y_test)
        loader = DataLoader(test_dataset, batch_size=32)

        self.model.eval()
        preds = []
        with torch.no_grad():
            for X_batch, _ in loader:
                outputs = self.model(X_batch)
                pred_labels = torch.argmax(outputs, dim=1)
                preds.extend(pred_labels.numpy())

        print(classification_report(y_test, preds))
        return classification_report(y_test, preds)


In [35]:
df = pd.read_excel("./sample_labelled/sample_labelled.xlsx", engine="openpyxl")

# Keywords to track
keywords = ["fraud", "money laundering", "corruption", "bribery",
            "sanction", 'ponzi', 'pyramid scheme',
            'insider trading', 'terrorist financing', 'tax-evasion']

# Columns you want to compute keyword density on
keyword_cols = ['title', 'description']

# Initialize the classifier
clf = BerTopicsEnhancedClassifier(
    df=df,
    label_col='adverse_news_financial_crime_scandal_sanctions',
    text_col='clean_full_text',
    keyword_cols=keyword_cols,
    keywords=keywords,
    use_tfidf=False
)

# Train the model
clf.train(epochs=50, batch_size=16, lr=3e-5)

# Evaluate it
bertopic_mlp_eval_df = clf.evaluate()


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 19/19 [00:01<00:00, 10.36it/s]


Epoch 1/50 - Train Loss: 0.7626 - Val Loss: 0.6963
Epoch 2/50 - Train Loss: 0.6960 - Val Loss: 0.6748
Epoch 3/50 - Train Loss: 0.6494 - Val Loss: 0.6504
Epoch 4/50 - Train Loss: 0.6113 - Val Loss: 0.6073
Epoch 5/50 - Train Loss: 0.5855 - Val Loss: 0.5904
Epoch 6/50 - Train Loss: 0.5593 - Val Loss: 0.5707
Epoch 7/50 - Train Loss: 0.5379 - Val Loss: 0.5639
Epoch 8/50 - Train Loss: 0.5196 - Val Loss: 0.5335
Epoch 9/50 - Train Loss: 0.5040 - Val Loss: 0.5313
Epoch 10/50 - Train Loss: 0.4947 - Val Loss: 0.5296
Epoch 11/50 - Train Loss: 0.4854 - Val Loss: 0.5207
Epoch 12/50 - Train Loss: 0.4750 - Val Loss: 0.5118
Epoch 13/50 - Train Loss: 0.4650 - Val Loss: 0.5094
Epoch 14/50 - Train Loss: 0.4559 - Val Loss: 0.5050
Epoch 15/50 - Train Loss: 0.4470 - Val Loss: 0.4999
Epoch 16/50 - Train Loss: 0.4413 - Val Loss: 0.4893
Epoch 17/50 - Train Loss: 0.4287 - Val Loss: 0.4913
Epoch 18/50 - Train Loss: 0.4309 - Val Loss: 0.4884
Epoch 19/50 - Train Loss: 0.4259 - Val Loss: 0.4902
Epoch 20/50 - Train L

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 19/19 [00:01<00:00, 11.12it/s]


              precision    recall  f1-score   support

           0       0.83      0.91      0.86       464
           1       0.52      0.35      0.42       136

    accuracy                           0.78       600
   macro avg       0.67      0.63      0.64       600
weighted avg       0.76      0.78      0.76       600



In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
from typing import List


# ---- Custom Transformers ----

class TextSelector(BaseEstimator, TransformerMixin):
    """
    Selects a single column from a DataFrame.
    """
    def __init__(self, column):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.column].fillna("")


class KeywordDensityExtractor(BaseEstimator, TransformerMixin):
    """
    Extracts keyword density features for a given list of keywords.
    """
    def __init__(self, keywords, column):
        self.keywords = keywords
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return pd.DataFrame([
            [text.lower().count(kw.lower()) / (len(text.split()) + 1e-6) for kw in self.keywords]
            for text in X[self.column].fillna("")
        ])


class TopicFeatureAdder(BaseEstimator, TransformerMixin):
    """
    Adds topic relevance features based on BERTopic model.
    """
    def __init__(self, topic_model: BERTopic, text_column: str, relevant_topic_ids: List[int]):
        self.topic_model = topic_model
        self.text_column = text_column
        self.relevant_topic_ids = relevant_topic_ids

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        texts = X[self.text_column].fillna("").tolist()
        topics, _ = self.topic_model.transform(texts)
        return pd.DataFrame([[int(t in self.relevant_topic_ids) for t in topics]], index=X.index).T.rename(columns={0: "topic_is_relevant"})


# ---- HuggingFace Dataset ----

class NewsDataset(Dataset):
    """
    Converts text and label data into a format compatible with HuggingFace Trainer.
    """
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


# ---- Classifier Pipeline ----

class TopicBERTAdverseClassifier:
    """
    Full classification pipeline including TF-IDF, keyword density, and optionally BERTopic features.
    """
    def __init__(self, df: pd.DataFrame, label_col: str, text_col: str = 'clean_full_text',
                 keyword_cols: List[str] = None, keywords: List[str] = None,
                 use_topic_features: bool = True):

        self.df = df.copy()
        self.label_col = label_col
        self.text_col = text_col
        self.keyword_cols = keyword_cols or ['title', 'description']
        self.keywords = keywords or ["fraud", "money laundering", "corruption", "bribery", "sanction"]
        self.use_topic_features = use_topic_features

        self.embedding_model = SentenceTransformer("all-MiniLM-L12-v2")
        self.df["topic_text"] = self.df[self.keyword_cols].fillna("").agg(" ".join, axis=1)
        self.topic_text_col = "topic_text"

        self._split_data()
        if self.use_topic_features:
            self._initialize_topic_model_with_relevance_filter()

    def _split_data(self):
        """
        Splits dataset into training, validation, and test sets.
        """
        train_val_df, self.test_df = train_test_split(
            self.df, test_size=0.2, stratify=self.df[self.label_col], random_state=42
        )
        self.train_df, self.val_df = train_test_split(
            train_val_df, test_size=0.25, stratify=train_val_df[self.label_col], random_state=42
        )

    def _initialize_topic_model_with_relevance_filter(self):
        """
        Initializes BERTopic and identifies relevant topics based on keyword similarity.
        """
        self.topic_model = BERTopic(embedding_model=self.embedding_model)
        current_train_texts = self.train_df[self.topic_text_col].fillna("").tolist()
        min_relevant = 5
        iteration = 0
        relevant_topic_ids = []

        while len(relevant_topic_ids) < min_relevant and iteration < 5:
            self.topic_model = BERTopic(embedding_model=self.embedding_model)
            topics, _ = self.topic_model.fit_transform(current_train_texts)
            topic_info = self.topic_model.get_topic_info()
            topic_ids = topic_info['Topic'].tolist()

            topic_representations = [self.topic_model.get_topic(t) for t in topic_ids if t != -1]
            topic_keywords = [" ".join([word for word, _ in rep[:5]]) for rep in topic_representations]

            topic_embeddings = self.embedding_model.encode(topic_keywords, convert_to_tensor=True)
            keyword_embeddings = self.embedding_model.encode(self.keywords, convert_to_tensor=True)

            similarities = util.cos_sim(topic_embeddings, keyword_embeddings)
            relevant_topic_ids = [topic_ids[i] for i in range(len(topic_ids))
                                  if i < similarities.shape[0] and similarities[i].max().item() > 0.4 and topic_ids[i] != -1]
            iteration += 1

        self.relevant_topic_ids = relevant_topic_ids

    def show_relevant_topics(self, top_n_words=10):
        """
        Display the top N words for each relevant topic ID.
        """
        if not self.relevant_topic_ids:
            print("No relevant topics found.")
            return

        topics_info = []
        for topic_id in self.relevant_topic_ids:
            topic_words = self.topic_model.get_topic(topic_id)
            if topic_words:
                top_words = ", ".join([word for word, _ in topic_words[:top_n_words]])
                topics_info.append((topic_id, top_words))

        topic_df = pd.DataFrame(topics_info, columns=["Topic ID", "Top Words"])
        return topic_df

    def show_topics(self, n=10):
        """
        Returns the top N topics from the BERTopic model.
        """
        return self.topic_model.get_topic_info().head(n)

    def get_feature_overview(self, df_subset='train'):
        """
        Returns preprocessed feature set for inspection.
        """
        df = {'train': self.train_df, 'val': self.val_df, 'test': self.test_df}[df_subset]
        return self.preprocess_features(df)

    def preprocess_features(self, df, fit=True):
        """
        Applies feature extraction using TF-IDF, keyword density, and optionally topic features.
        """
        feature_extractors = []

        feature_extractors.append((
            "tfidf",
            Pipeline([
                ('select', TextSelector(self.text_col)),
                ('tfidf', TfidfVectorizer(max_features=3000, stop_words='english'))
            ])
        ))

        for col in self.keyword_cols:
            feature_extractors.append((
                f"{col}_keyword_density",
                KeywordDensityExtractor(self.keywords, column=col)
            ))

        if self.use_topic_features:
            feature_extractors.append((
                "topic",
                TopicFeatureAdder(self.topic_model, text_column=self.topic_text_col,
                                  relevant_topic_ids=self.relevant_topic_ids)
            ))

        self.feature_pipeline = FeatureUnion(feature_extractors)
        return self.feature_pipeline.fit_transform(df) if fit else self.feature_pipeline.transform(df)

    def train_transformer_classifier(self, num_train_epochs=30, model_name='distilbert-base-uncased', output_dir='./clf_model'):
        """
        Fine-tunes a HuggingFace transformer model on the training data.
        """
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

        self.train_dataset = NewsDataset(
            texts=self.train_df[self.text_col].tolist(),
            labels=self.train_df[self.label_col].tolist(),
            tokenizer=self.tokenizer
        )

        self.val_dataset = NewsDataset(
            texts=self.val_df[self.text_col].tolist(),
            labels=self.val_df[self.label_col].tolist(),
            tokenizer=self.tokenizer
        )

        self.test_dataset = NewsDataset(
            texts=self.test_df[self.text_col].tolist(),
            labels=self.test_df[self.label_col].tolist(),
            tokenizer=self.tokenizer
        )

        training_args = TrainingArguments(
            output_dir=output_dir,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_dir=f'{output_dir}/logs',
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=num_train_epochs,
            weight_decay=0.01,
            learning_rate = 1e-5,
            logging_steps=10,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            save_total_limit=2,
        )

        def compute_metrics(pred):
            labels = pred.label_ids
            preds = np.argmax(pred.predictions, axis=1)
            proba = pred.predictions[:, 1]
            return {
                "accuracy": accuracy_score(labels, preds),
                "precision": precision_score(labels, preds),
                "recall": recall_score(labels, preds),
                "f1": f1_score(labels, preds),
                "roc_auc": roc_auc_score(labels, proba),
            }

        self.trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.val_dataset,
            compute_metrics=compute_metrics
        )

        self.trainer.train()
        return self.trainer

    def evaluate_on_test_set(self, trainer=None):
        """
        Evaluates the model on the test dataset.
        """
        trainer = trainer or self.trainer
        print("\n--- Evaluating on Test Set ---")
        predictions = trainer.predict(self.test_dataset)
        preds = np.argmax(predictions.predictions, axis=1)
        probs = predictions.predictions[:, 1]
        labels = predictions.label_ids

        return pd.DataFrame([{
            "accuracy": accuracy_score(labels, preds),
            "precision": precision_score(labels, preds),
            "recall": recall_score(labels, preds),
            "f1": f1_score(labels, preds),
            "roc_auc": roc_auc_score(labels, probs)
        }])


# ---------- Run Training ----------

if __name__ == '__main__':
    df = pd.read_excel("./sample_labelled/sample_labelled.xlsx", engine="openpyxl")

    ## The following implements with BerTopics features
    classifier = TopicBERTAdverseClassifier(
        df=df,
        label_col='adverse_news_financial_crime_scandal_sanctions',
        text_col='clean_full_text',
        keyword_cols=['title', 'description'],
        keywords=[
            "fraud", "money laundering", "corruption", "bribery",
            "sanction", 'ponzi', 'pyramid scheme',
            'insider trading', 'terrorist financing', 'tax-evasion'
        ],
        use_topic_features=True  # Set to False to exclude BERTopic features
    )

    print("\nRelevant Topics with Top Words:")
    print(classifier.show_relevant_topics())

    trainer = classifier.train_transformer_classifier()
    print("Training complete. Best model and logs saved.")

    test_eval_bertpipeline = classifier.evaluate_on_test_set(trainer)
    print("\nTest Set Evaluation (with Topic Modelling):")
    print(test_eval_bertpipeline)

    ## The following doesn't include BerTopic Features
    no_topic_classifier = TopicBERTAdverseClassifier(
        df=df,
        label_col='adverse_news_financial_crime_scandal_sanctions',
        text_col='clean_full_text',
        keyword_cols=['title', 'description'],
        keywords=[
            "fraud", "money laundering", "corruption", "bribery",
            "sanction", 'ponzi', 'pyramid scheme',
            'insider trading', 'terrorist financing', 'tax-evasion'
        ],
        use_topic_features=False  # Set to False to exclude BERTopic features
    )

    no_topic_trainer = no_topic_classifier.train_transformer_classifier()
    print("Training complete. Best model and logs saved.")

    test_eval_bertpipeline_no_topic = no_topic_classifier.evaluate_on_test_set(no_topic_trainer)
    print("\nTest Set Evaluation (without BerTopics):")
    print(test_eval_bertpipeline_no_topic)


Relevant Topics with Top Words:
   Topic ID                                          Top Words
0         4  kim, hyun, soo, ron, sae, scandal, dating, act...
1         5  efcc, nigeria, laundering, money, nigerian, go...
2         6  scam, scams, alert, warning, smishing, toll, p...
3        13  musk, elon, social, security, jamaal, bowman, ...
4        14  tax, hmrc, evasion, income, chosunbiz, billion...
5        19  trump, tariffs, tariff, business, doge, inside...
6        22  globenewswire, alert, grossman, bronstein, gew...
7        23  buying, insider, monkey, insiders, stocks, mar...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.553200,0.514244,0.775000,0.000000,0.000000,0.000000,0.653493
2,0.478600,0.494205,0.775000,0.000000,0.000000,0.000000,0.701752
3,0.405100,0.497073,0.790000,0.714286,0.111111,0.192308,0.708292
4,0.385300,0.508268,0.758333,0.443182,0.288889,0.349776,0.711454
5,0.279500,0.559237,0.741667,0.410714,0.340741,0.372470,0.710466
6,0.184900,0.684870,0.786667,0.550725,0.281481,0.372549,0.705352
7,0.260400,0.770447,0.751667,0.440678,0.385185,0.411067,0.697308
8,0.182200,0.819297,0.756667,0.439560,0.296296,0.353982,0.701099
9,0.072600,0.914758,0.753333,0.431579,0.303704,0.356522,0.696241
10,0.156800,1.086081,0.726667,0.392593,0.392593,0.392593,0.691509


Training complete. Best model and logs saved.

--- Evaluating on Test Set ---



Test Set Evaluation (with Topic Modelling):
   accuracy  precision    recall        f1   roc_auc
0  0.758333   0.467626  0.477941  0.472727  0.716959


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.551700,0.520089,0.775000,0.000000,0.000000,0.000000,0.638144
2,0.495200,0.504500,0.775000,0.000000,0.000000,0.000000,0.681601
3,0.396100,0.508269,0.783333,0.857143,0.044444,0.084507,0.691979
4,0.383400,0.515005,0.741667,0.380952,0.237037,0.292237,0.704221
5,0.316400,0.576260,0.760000,0.421053,0.177778,0.250000,0.685830
6,0.200800,0.662876,0.776667,0.506849,0.274074,0.355769,0.676886
7,0.228300,0.726646,0.753333,0.442478,0.370370,0.403226,0.696241
8,0.213000,0.822623,0.755000,0.437500,0.311111,0.363636,0.698773
9,0.121600,0.884773,0.765000,0.448276,0.192593,0.269430,0.663879
10,0.172700,0.965559,0.756667,0.445545,0.333333,0.381356,0.674377


Training complete. Best model and logs saved.

--- Evaluating on Test Set ---



Test Set Evaluation (without BerTopics):
   accuracy  precision    recall        f1   roc_auc
0  0.758333    0.46087  0.389706  0.422311  0.748637
